<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/03-From_Script_to_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Script to Pipeline: Chunking and Reusable Retrieval Functions

The from-scratch RAG script from the last notebook worked — but it was a *script*: variables flowing into variables, impossible to reuse or test in pieces. In this notebook we refactor it into the four named functions the whole course builds on — **`chunk()` → `embed()` → `search()` → `answer()`** — and we finally take chunking seriously: token-based chunks with overlap, and a mini version of the **production heading-aware chunker** that never cuts a code block in half.

📎 *These are the same function names and responsibilities as the production AI Tutor's pipeline. From here to the end of Part 1, every improvement is an upgrade to one of these four functions — the interfaces stay put.*

## 🧭 What You'll Learn

- Why chunk sizes are measured in **tokens** (not characters) and counted with a real tokenizer
- `chunk()`: token-based splitting with **overlap**, and why overlap prevents severed facts
- A mini **heading-aware Markdown chunker** (800 tokens / 100 overlap) that respects structure and never splits code blocks — the production tutor's approach
- `search()` and `answer()`: retrieval and generation as small, testable functions
- A **chunk-size sweep**: how retrieval quality visibly changes at 256 vs 512 vs 1024 tokens

## 1. Setup: Environment, Keys, and Providers

The standard course setup cell — provider and model pickers, pinned installs (now including `tiktoken` for token counting), keys from Colab Secrets or a local `.env`.

The **model field is an editable dropdown** (`{allow-input: true}`): pick one of the listed course defaults, or type any newer model ID straight into the box — no code changes needed. (Locally, simply edit the string.)

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model providers (dropdowns in Colab; edit the values locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]
EMBED_PROVIDER = "gemini"  # @param ["gemini", "openai"]  (Anthropic has no embedding API)

# Pick a model for the selected provider — or TYPE any newer model ID into the
# box (the dropdown is editable thanks to allow-input):
CHAT_MODEL = "gemini-3.7-flash"  # @param ["gemini-3.7-flash", "gemini-3.5-flash-lite", "gpt-5.6-luna", "claude-sonnet-5"] {allow-input: true}
EMBED_MODEL = "gemini-embedding-001"  # @param ["gemini-embedding-001", "text-embedding-3-small"] {allow-input: true}

_KEY_FOR = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
REQUIRED_KEYS = sorted({_KEY_FOR[PROVIDER], _KEY_FOR[EMBED_PROVIDER]})

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (July 2026).
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "google-genai==2.3.0",
            "openai==2.46.0",
            "anthropic==0.117.0",
            "tiktoken==0.13.0",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

    # In Colab: Secrets tab (🔑 icon) → Add new secret → e.g. GOOGLE_API_KEY
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    # Locally: dependencies are installed once from the repo's requirements.
    # Keys live in a .env file at the repo root.
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | chat: {PROVIDER} | embeddings: {EMBED_PROVIDER}")

✅ Setup complete — local | chat: gemini | embeddings: gemini


## 2. `generate()` and `embed()`

📎 *Unchanged from the Basic RAG notebook — the course's two provider helpers.*

In [2]:
# 📎 generate() was built in the "How To Use LLMs via API" notebook.
#    embed() is NEW in this notebook — and reused for the rest of the course.
from anthropic import Anthropic
from google import genai
from google.genai import types as genai_types
from openai import OpenAI

# Course-standard default models per provider (August 2026)
MODELS = {
    "gemini": "gemini-3.7-flash",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-sonnet-5",
}

# The setup-cell form selection (or any typed model ID) overrides the default:
MODELS[PROVIDER] = CHAT_MODEL
EMBED_MODELS = {
    "gemini": "gemini-embedding-001",
    "openai": "text-embedding-3-small",
}
EMBED_MODELS[EMBED_PROVIDER] = EMBED_MODEL  # setup-cell selection (or typed ID) wins
EMBED_DIM = 1536  # same output size for both providers, so the rest of the code never cares

# Create only the clients we actually need
if "gemini" in (PROVIDER, EMBED_PROVIDER):
    gemini_client = genai.Client()
if "openai" in (PROVIDER, EMBED_PROVIDER):
    openai_client = OpenAI()
if PROVIDER == "anthropic":
    anthropic_client = Anthropic()


def generate(prompt, system=None, model=None):
    """Send one prompt to the selected PROVIDER and return the reply text."""
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(system_instruction=system),
        )
        return response.text

    if PROVIDER == "openai":
        response = openai_client.responses.create(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            reasoning={"effort": "none"},
        )
        return response.output_text

    if PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=model or MODELS["anthropic"],
            max_tokens=4096,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def embed(texts, task="document"):
    """Embed a list of texts with the selected EMBED_PROVIDER.

    Returns a list of EMBED_DIM-dimensional vectors (one per input text).
    task: "document" for corpus chunks, "query" for user questions —
    Gemini embeds the two slightly differently to improve retrieval.
    """
    if isinstance(texts, str):
        texts = [texts]
    # Embedding models work best on single-line inputs
    texts = [t.replace("\n", " ") for t in texts]

    if EMBED_PROVIDER == "gemini":
        result = gemini_client.models.embed_content(
            model=EMBED_MODELS["gemini"],
            contents=texts,
            config=genai_types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT" if task == "document" else "RETRIEVAL_QUERY",
                output_dimensionality=EMBED_DIM,
            ),
        )
        return [e.values for e in result.embeddings]

    if EMBED_PROVIDER == "openai":
        result = openai_client.embeddings.create(
            model=EMBED_MODELS["openai"], input=texts
        )
        return [d.embedding for d in result.data]

    raise ValueError(f"Unknown EMBED_PROVIDER: {EMBED_PROVIDER!r}")

## 3. Load the Dataset — Keeping the Metadata This Time

Same corpus as before (the course's **ai-docs** set: 23 recent AI articles — model cards, release announcements, and technical explainers), with one upgrade: we keep each article's **title, URL, and source** alongside its text. Metadata rides along with every chunk from now on — the next lessons use it for filtering and source citations.

In [3]:
import csv
import pathlib

import requests

# Course dataset — hosted in the Towards AI org dataset repo on Hugging Face
DATA_URL = "https://huggingface.co/datasets/towardsai-tutors/full-stack-ai-engineering-data/resolve/main/ai-docs.csv"
DATA_PATH = pathlib.Path("ai-docs.csv")

if not DATA_PATH.exists():
    DATA_PATH.write_bytes(requests.get(DATA_URL, timeout=30).content)
    print("Downloaded", DATA_PATH)

# Read the articles WITH their metadata (title, url, source) — we carry it everywhere
articles = []
with open(DATA_PATH, mode="r", encoding="utf-8") as f:
    for idx, row in enumerate(csv.reader(f)):
        if idx == 0:
            continue  # skip the header row
        articles.append({"title": row[0], "text": row[1], "url": row[2], "source": row[3]})

print(f"{len(articles)} articles loaded")

23 articles loaded


## 4. `chunk()` — Token-Based Splitting with Overlap

Two upgrades over the naive 1024-*character* splitter from the last notebook:

1. **Tokens, not characters.** Models and embedding APIs bill, truncate, and attend in tokens. A 512-token chunk is a meaningful unit; a 1024-character chunk is roughly 250 tokens of unknown alignment. We count with `tiktoken`.
2. **Overlap.** A hard cut can sever a fact ("Qwen3.8 has 2.4T parameters in total and ⟪cut⟫ 95B activated"). Sharing 128 tokens between consecutive chunks means every fact appears *whole* in at least one chunk.

In [4]:
import tiktoken

# The tokenizer used for counting: chunk sizes in this course are measured in TOKENS
ENC = tiktoken.get_encoding("cl100k_base")


def n_tokens(text):
    """Number of tokens in a string."""
    return len(ENC.encode(text))


def chunk(text, chunk_size=512, chunk_overlap=128):
    """Split text into token-based chunks; consecutive chunks share chunk_overlap tokens."""
    tokens = ENC.encode(text)
    chunks = []
    step = chunk_size - chunk_overlap
    for start in range(0, len(tokens), step):
        window = tokens[start : start + chunk_size]
        chunks.append(ENC.decode(window))
        if start + chunk_size >= len(tokens):
            break  # the final window reached the end of the text
    return chunks

In [5]:
# Chunk every article, carrying its metadata onto every chunk it produces
records = []
for art in articles:
    for i, piece in enumerate(chunk(art["text"], chunk_size=512, chunk_overlap=128)):
        records.append({
            "id": f"{art['title'][:40]}-{i}",
            "text": piece,
            "title": art["title"],
            "url": art["url"],
            "source": art["source"],
        })

print(f"{len(articles)} articles → {len(records)} chunks")
print("Example boundary — note the shared text between consecutive chunks:")
print("…", records[0]["text"][-120:].replace("\n", " "))
print("↕ overlap ↕")
print(records[1]["text"][:120].replace("\n", " "), "…")

23 articles → 233 chunks
Example boundary — note the shared text between consecutive chunks:
… g Agent | | | | | | | Terminal Bench 2.1 | 84.6 | 84.6 | 88.8 | 74.5 | 86.6 | | SWE-bench Pro | 69.2 | 80.0 | 64.6 | 60.
↕ overlap ↕
144 natively and extensible up to 1,010,000 tokens.  ## Benchmark Results  | | Opus 4.8 | Fable 5 | GPT 5.6 Sol (max) |  …


**What just happened?** `chunk()` turned each article into overlapping ~512-token windows, and every chunk kept its parent article's metadata. Look at the printed boundary: the end of chunk 0 reappears at the start of chunk 1 — that duplication is deliberate insurance against severed facts.

## 5. The Production Chunker: Heading-Aware, Code-Safe

Fixed-size windows ignore document *structure*: they happily cut a Markdown section in the middle of a sentence, or worse, split a code block in half (instant syntax garbage in retrieval results). The production AI Tutor indexes documentation, so its chunker works differently: **split at Markdown headings, pack whole sections into chunks of at most 800 tokens with 100-token overlap, and never break a fenced code block.** Here is a mini version:

In [6]:
def heading_aware_markdown_chunks(markdown, chunk_size=800, chunk_overlap=100):
    """Mini version of the production tutor's chunker.

    Splits a Markdown document at headings, packs whole sections into chunks of
    at most chunk_size tokens (carrying chunk_overlap tokens between chunks),
    and NEVER splits a fenced code block in half.
    """
    # --- 1) Cut the document into blocks: a heading starts a new block;
    #        fenced code (```) is glued to its block no matter how long.
    blocks, current, in_code = [], [], False
    for line in markdown.splitlines(keepends=True):
        if line.lstrip().startswith("```"):
            in_code = not in_code
            current.append(line)
            continue
        if not in_code and line.startswith("#") and current:
            blocks.append("".join(current))
            current = [line]
        else:
            current.append(line)
    if current:
        blocks.append("".join(current))

    # --- 2) Pack whole blocks into chunks of at most chunk_size tokens.
    chunks, buf = [], ""
    for block in blocks:
        if n_tokens(block) > chunk_size and "```" not in block:
            # Oversized prose block: flush the buffer, fall back to token chunking.
            if buf.strip():
                chunks.append(buf)
                buf = ""
            chunks.extend(chunk(block, chunk_size, chunk_overlap))
        elif buf and n_tokens(buf) + n_tokens(block) > chunk_size:
            # Buffer is full: emit it, carry the last chunk_overlap tokens forward.
            chunks.append(buf)
            overlap_text = ENC.decode(ENC.encode(buf)[-chunk_overlap:])
            buf = overlap_text + "\n" + block
        else:
            buf += block
    if buf.strip():
        chunks.append(buf)
    return chunks

In [7]:
# Demo on a small Markdown document with headings AND a code block
SAMPLE_DOC = """# Calling the Tutor API

The tutor exposes one endpoint for questions.

## Authentication

Requests carry a bearer token in the Authorization header.

## Example request

```python
import requests

r = requests.post(
    "https://tutor.example.com/ask",
    headers={"Authorization": f"Bearer {TOKEN}"},
    json={"question": "What is RRF?"},
)
print(r.json()["answer"])
```

## Rate limits

Sixty requests per minute per token. Beyond that the API returns HTTP 429.
"""

for i, c in enumerate(heading_aware_markdown_chunks(SAMPLE_DOC, chunk_size=120, chunk_overlap=20)):
    fences = c.count("```")
    print(f"┌─ chunk {i} | {n_tokens(c)} tokens | code fences: {fences} ({'intact ✅' if fences % 2 == 0 else 'SPLIT ❌'})")
    print("│ " + c.strip().replace("\n", "\n│ ")[:400])
    print("└" + "─" * 60)

┌─ chunk 0 | 107 tokens | code fences: 2 (intact ✅)
│ # Calling the Tutor API
│ 
│ The tutor exposes one endpoint for questions.
│ 
│ ## Authentication
│ 
│ Requests carry a bearer token in the Authorization header.
│ 
│ ## Example request
│ 
│ ```python
│ import requests
│ 
│ r = requests.post(
│     "https://tutor.example.com/ask",
│     headers={"Authorization": f"Bearer {TOKEN}"},
│     json={"question": "What is RRF?"},
│ )
│ print(r.json()["ans
└────────────────────────────────────────────────────────────


**What just happened?** We forced a tiny 120-token budget to make the chunker sweat — and it still emitted chunks that start at headings and keep the code block in one piece (every chunk has an *even* number of fence markers). Our article corpus is plain prose, so the pipeline below keeps using `chunk()`; the moment the course ingests real Markdown documentation, `heading_aware_markdown_chunks()` takes over — with the production settings of **800 tokens and 100 overlap**.

## 6. `search()` — Retrieval as a Function

Embed the corpus once, then wrap the embed-question → cosine-scores → top-k dance from the last notebook into a single function that returns chunks *with their scores and metadata*.

In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

df = pd.DataFrame(records)

# Embed the whole knowledge base in batches (same as the previous notebook)
BATCH_SIZE = 50
embeddings = []
for start in tqdm(range(0, len(df), BATCH_SIZE)):
    embeddings.extend(embed(df["text"].iloc[start : start + BATCH_SIZE].tolist(), task="document"))
df["embedding"] = embeddings

EMB_MATRIX = np.stack(df["embedding"].to_numpy())
print("Knowledge base:", EMB_MATRIX.shape)

  0%|          | 0/5 [00:00<?, ?it/s]

Knowledge base: (233, 1536)


In [9]:
def search(query, top_k=5):
    """Return the top_k most relevant chunks for a query: text, score, metadata."""
    q = np.array(embed(query, task="query")[0])
    scores = cosine_similarity([q], EMB_MATRIX)[0]
    top = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "text": df["text"][i],
            "score": float(scores[i]),
            "title": df["title"][i],
            "url": df["url"][i],
        }
        for i in top
    ]


# Always LOOK at retrieval output before wiring it into generation
for r in search("How many parameters does the Qwen3.8 model have?", top_k=3):
    print(f"┌─ {r['score']:.4f} | {r['title'][:60]}")
    print(f"│  {r['text'][:200]}".replace("\n", " "))
    print("└" + "─" * 60)

┌─ 0.7828 | Qwen3.8-2.4T-A95B Model Card
│  # Qwen3.8-2.4T-A95B  ## Qwen3.8 Highlights  Qwen3.8 features the following enhancements:  - Core Capabilities: Comprehensive improvements across coding, professional work, research, and long-horizon a
└────────────────────────────────────────────────────────────
┌─ 0.7407 | Qwen3.8-2.4T-A95B Model Card
│  144 natively and extensible up to 1,010,000 tokens.  ## Benchmark Results  | | Opus 4.8 | Fable 5 | GPT 5.6 Sol (max) | Qwen3.7-Max | Qwen3.8-Max |  | Coding Agent | | | | | | | Terminal Bench 2.1 | 8
└────────────────────────────────────────────────────────────
┌─ 0.7299 | Qwen3.8-2.4T-A95B Model Card
│   Usage  Qwen3.8 comes with official support for `reasoning_effort`, which can be used to adjust reasoning depth and control cost:  - `xhigh` (default): for complex tasks demanding thorough analysis - 
└────────────────────────────────────────────────────────────


## 7. `answer()` — Generation as a Function

The course augmentation template (unchanged since RAG 101), wrapped so that the caller just passes a question and retrieved chunks.

In [10]:
SYSTEM_INSTRUCTION = (
    "You are an assistant and expert in answering questions from a chunks of content. "
    "Only answer AI-related question, else say that you cannot answer this question."
)

PROMPT_TEMPLATE = (
    "Read the following informations that might contain the context you require to "
    "answer the question. You can use the informations starting from the "
    "<START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n"
    "<START_OF_CONTEXT>\n{context}\n<END_OF_CONTEXT>\n\n"
    "Please provide an informative and accurate answer to the following question based "
    "on the available context. Be concise and take your time.\nQuestion: {question}\nAnswer:"
)


def answer(question, top_k=5):
    """The whole pipeline in one call: search, then grounded generation."""
    retrieved = search(question, top_k=top_k)
    context = "\n\n".join(r["text"] for r in retrieved)
    reply = generate(
        PROMPT_TEMPLATE.format(context=context, question=question),
        system=SYSTEM_INSTRUCTION,
    )
    return reply, retrieved


reply, retrieved = answer("How many parameters does the Qwen3.8 model have?")
print(reply)

Based on the provided context, the Qwen3.8 model has a total of **2.4 trillion (2.4T)** parameters, with **95 billion (95B)** activated parameters.


**What just happened?** The entire RAG system is now four named functions with clean seams: `chunk()` (text → pieces), `embed()` (pieces → vectors), `search()` (question → best pieces), `answer()` (best pieces → grounded reply). This is the exact shape of the production tutor's pipeline — everything the rest of Part 1 does (vector databases, hybrid search, reranking, evaluation) upgrades one seam without touching the others.

## 8. The Chunk-Size Sweep: Seeing the Trade-off

Is 512 tokens *right*? Chunk size is the first knob every RAG builder over- or under-tunes. Small chunks are precise but starve the model of context; big chunks carry context but dilute the embedding (many topics averaged into one vector). Let's rebuild the knowledge base at three sizes and watch the same question behave differently.

💡 *This re-embeds the corpus three times — a few cents, and a taste of the systematic evaluation coming in the RAG-evaluation lesson, where sweeps like this get real metrics instead of eyeballs.*

In [11]:
QUESTION = "How many parameters does the Qwen3.8 model have?"
EXPECTED_FACT = "95B activated"  # stated once, only in the Qwen3.8 card — a cheap proxy for "the right chunk came back"

for size, overlap in [(256, 32), (512, 128), (1024, 128)]:
    texts = [piece for art in articles for piece in chunk(art["text"], size, overlap)]
    vecs = []
    for start in range(0, len(texts), 50):
        vecs.extend(embed(texts[start : start + 50], task="document"))
    matrix = np.stack(vecs)

    q = np.array(embed(QUESTION, task="query")[0])
    scores = cosine_similarity([q], matrix)[0]
    top = np.argsort(scores)[::-1][:3]
    hit = any(EXPECTED_FACT in texts[i] for i in top)

    print(f"chunk_size={size:5d} | chunks={len(texts):4d} | top-3 scores="
          f"{[round(float(scores[i]), 3) for i in top]} | expected fact in top-3: {'✅' if hit else '❌'}")

chunk_size=  256 | chunks= 398 | top-3 scores=[0.78, 0.763, 0.725] | expected fact in top-3: ✅


chunk_size=  512 | chunks= 233 | top-3 scores=[0.783, 0.741, 0.73] | expected fact in top-3: ✅


chunk_size= 1024 | chunks= 107 | top-3 scores=[0.773, 0.725, 0.684] | expected fact in top-3: ✅


**What just happened?** Same corpus, same question — different chunk sizes produce different chunk counts, different similarity scores, and sometimes a different verdict on whether the right fact surfaces at all. Notice the pattern: scores are not comparable *across* configurations, so "which is better" needs a proper metric over many questions — exactly what hit rate and MRR will give us in the evaluation lesson. Until then, the course default is **512/128 for prose** and **800/100 heading-aware for documentation**.

## 🔑 Key Takeaways

- The course pipeline is four functions — **`chunk()` → `embed()` → `search()` → `answer()`** — with stable interfaces; every later lesson upgrades one of them in place.
- Measure chunks in **tokens** (`tiktoken`), not characters, and always use **overlap** so no fact is severed at a boundary.
- Structure-aware chunking beats fixed windows for real documents: the production chunker splits at Markdown headings, packs to **800 tokens with 100 overlap**, and never breaks a code block.
- Retrieval functions should return **scores and metadata**, not bare strings — you cannot debug what you cannot see.
- Chunk size is a measurable trade-off, not a vibe: sweeps need per-question metrics (hit rate, MRR), which arrive in the RAG-evaluation lesson.